# EDA: Ethiopia Financial Inclusion
This notebook performs schema exploration and expanded EDA on the enriched dataset.
Objectives:
- Inspect schema and data quality
- Visualize temporal coverage and key indicators
- Overlay events and examine preliminary relationships
- Produce figures saved to `reports/figures/` for reporting

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from pathlib import Path

sns.set(style='whitegrid')
DATA_PATH = Path('..') / 'data' / 'processed' / 'ethiopia_fi_unified_data_enriched.csv'
FIG_DIR = Path('..') / 'reports' / 'figures'
FIG_DIR.mkdir(parents=True, exist_ok=True)
df = pd.read_csv(DATA_PATH)
df['observation_date'] = pd.to_datetime(df['observation_date'], errors='coerce')
df['value_numeric'] = pd.to_numeric(df['value_numeric'], errors='coerce')
df.head()

In [ ]:
# Schema inspection
print('Rows, cols:', df.shape)
print('Columns:', list(df.columns))
print('
Dtypes:')
print(df.dtypes)

# Missingness by column
missing = df.isnull().mean().sort_values(ascending=False)
print('
Missingness (fraction):')
print(missing)

In [ ]:
# Counts by key categorical fields
print('
Record type counts:')
print(df['record_type'].value_counts(dropna=False))
print('
Pillar counts:')
print(df['pillar'].value_counts(dropna=False))
print('
Source counts:')
print(df['source_name'].value_counts(dropna=False).head(10))
print('
Confidence distribution:')
print(df['confidence'].value_counts(dropna=False))

In [ ]:
# Indicator coverage and temporal range
indicators = df['indicator_code'].dropna().unique()
print('Unique indicators (count):', len(indicators))
print(indicators)

# temporal range per indicator
df_obs = df[df['record_type']=='observation'].copy()
df_obs['year'] = pd.DatetimeIndex(df_obs['observation_date']).year
range_table = df_obs.groupby('indicator_code')['year'].agg(['min','max','count']).sort_values('count', ascending=False)
print(range_table)

In [ ]:
# Temporal coverage heatmap (year x indicator presence)
pivot = df_obs.pivot_table(index='year', columns='indicator_code', values='value_numeric', aggfunc='count').fillna(0)
plt.figure(figsize=(10,6))
sns.heatmap((pivot>0).astype(int), cmap='Blues', cbar=False)
plt.title('Temporal coverage: year vs indicator (presence)')
plt.tight_layout()
plt.savefig(FIG_DIR / 'temporal_coverage_heatmap.png')
plt.show()

In [ ]:
# Account Ownership over time (Access)
acc = df_obs[df_obs['indicator_code']=='account_ownership'].sort_values('observation_date')
plt.figure(figsize=(8,4))
sns.lineplot(data=acc, x='observation_date', y='value_numeric', marker='o')
plt.title('Account Ownership (Global Findex)')
plt.ylabel('Percent of adults')
plt.xlabel('Date')
plt.tight_layout()
plt.savefig(FIG_DIR / 'account_ownership.png')
plt.show()

In [ ]:
# Digital payments and mobile subscribers trend
dp = df_obs[df_obs['indicator_code']=='digital_payments']
ms = df_obs[df_obs['indicator_code']=='mobile_subscribers']
plt.figure(figsize=(8,4))
if not dp.empty:
    sns.lineplot(data=dp, x='observation_date', y='value_numeric', marker='o', label='Digital payments')
if not ms.empty:
    sns.lineplot(data=ms, x='observation_date', y='value_numeric', marker='o', label='Mobile subscribers (M)')
plt.title('Digital payments and mobile subscriber trends')
plt.ylabel('Value')
plt.xlabel('Date')
plt.legend()
plt.tight_layout()
plt.savefig(FIG_DIR / 'digital_payments_mobile_subscribers.png')
plt.show()

In [ ]:
# Event timeline overlay
events = df[df['record_type']=='event'].copy()
events = events[pd.notna(events['observation_date'])]
plt.figure(figsize=(10,2))
y = [1]*len(events)
plt.scatter(events['observation_date'], y)
for i, (_, e) in enumerate(events.iterrows()):
    plt.text(e['observation_date'], 1.02, (e.get('notes') or '')[:60], rotation=45, fontsize=8)
plt.gca().get_yaxis().set_visible(False)
plt.title('Event timeline')
plt.tight_layout()
plt.savefig(FIG_DIR / 'event_timeline.png')
plt.show()

In [ ]:
# Correlation analysis across indicators (observations only)
pivot = df_obs.pivot_table(index='observation_date', columns='indicator_code', values='value_numeric', aggfunc='mean')
corr = pivot.corr()
plt.figure(figsize=(9,7))
sns.heatmap(corr, annot=True, fmt='.2f', cmap='vlag')
plt.title('Correlation matrix (observed indicators)')
plt.tight_layout()
plt.savefig(FIG_DIR / 'correlation_matrix.png')
plt.show()

## Key Insights (preliminary)
- Account ownership increased from ~46% (2021) to ~49% (2024) — modest growth despite mobile money expansion.
- Digital payment adoption (~35% in 2024) is high relative to account ownership, suggesting active mobile payment use possibly outside formal accounts.
- Mobile subscribers (~65M) show broad reach but conversion to active account usage appears limited.
- 4G coverage (~45%) and low agent density (~8 agents/100k) indicate infrastructure constraints for rural on-boarding and merchant payments.
- Events (Telebirr, M-Pesa) align temporally with indicator upticks; these are hypotheses for impact modeling.

## Data Limitations and Next Steps
- Global Findex is triennial: low temporal frequency limits causal attribution.
- Missing disaggregations (gender, region) prevent equity analysis; seek Findex microdata or operator breakdowns.
- Operator registration vs survey activity mismatch: request active-account metrics from operators.
- Next: run impact models using `impact_link` records, produce scenario forecasts for 2025–2027, and add uncertainty bands.